# Part 4 — Vector Databases: Embeddings & Semantic Similarity

**Model:** `all-MiniLM-L6-v2` from `sentence-transformers`  
**Topics:** Cricket, Cooking, Cybersecurity (10 sentences total)  
**Tasks:** Cosine similarity matrix heatmap + top-2 similar sentences for a query

In [ ]:
# Install dependencies (run once in Colab)
!pip install sentence-transformers seaborn matplotlib --quiet

In [ ]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

print('Libraries loaded successfully.')

## Step 1 — Define 10 Sentences Across 3 Topics

In [ ]:
sentences = [
    # Cricket (indices 0–3)
    "The batsman hit a stunning century in the final over.",
    "India won the match by six wickets in the last delivery.",
    "The spinner took five wickets to turn the game around.",
    "The cricket pitch was damp after overnight rain, favouring swing bowling.",

    # Cooking (indices 4–6)
    "Sauté the onions in olive oil until they turn golden brown.",
    "Marinating the chicken overnight in yoghurt makes it tender and juicy.",
    "A pinch of garam masala added at the end elevates the entire dish.",

    # Cybersecurity (indices 7–9)
    "A zero-day exploit was used to bypass the corporate firewall.",
    "Multi-factor authentication significantly reduces the risk of account takeover.",
    "Phishing emails often impersonate trusted institutions to steal credentials."
]

labels = [
    'C1: Batsman century',
    'C2: India won six wickets',
    'C3: Spinner five wickets',
    'C4: Damp pitch swing',
    'K1: Saute onions',
    'K2: Marinate chicken',
    'K3: Garam masala',
    'S1: Zero-day exploit',
    'S2: MFA reduces risk',
    'S3: Phishing emails'
]

print(f'Total sentences: {len(sentences)}')
for i, s in enumerate(sentences):
    print(f'  [{i}] {s}')

## Step 2 — Generate Embeddings

In [ ]:
model = SentenceTransformer('all-MiniLM-L6-v2')
print('Model loaded: all-MiniLM-L6-v2')

embeddings = model.encode(sentences, show_progress_bar=True)
print(f'Embedding shape: {embeddings.shape}')  # Expected: (10, 384)

## Step 3 — Compute 10×10 Cosine Similarity Matrix

In [ ]:
sim_matrix = cosine_similarity(embeddings)
print('Cosine similarity matrix (10x10):')
print(np.round(sim_matrix, 3))

## Step 4 — Display Heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))

sns.heatmap(
    sim_matrix,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    xticklabels=labels,
    yticklabels=labels,
    vmin=0,
    vmax=1,
    linewidths=0.5,
    ax=ax
)

ax.set_title('Cosine Similarity Matrix — Cricket | Cooking | Cybersecurity', fontsize=14, pad=15)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.savefig('similarity_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('Heatmap saved as similarity_heatmap.png')

## Step 5 — Query Similarity: "The bowler took three wickets in one over"

In [ ]:
query = "The bowler took three wickets in one over"

query_embedding = model.encode([query])
query_similarities = cosine_similarity(query_embedding, embeddings)[0]

# Rank sentences by similarity score
ranked_indices = np.argsort(query_similarities)[::-1]

print(f'Query: "{query}"')
print('=' * 60)
print('\nTop 2 Most Similar Sentences:\n')

for rank, idx in enumerate(ranked_indices[:2], start=1):
    print(f'Rank {rank}:')
    print(f'  Sentence : {sentences[idx]}')
    print(f'  Label    : {labels[idx]}')
    print(f'  Similarity Score: {query_similarities[idx]:.4f}')
    print()

print('\nFull similarity scores for all sentences:')
for idx in ranked_indices:
    print(f'  {query_similarities[idx]:.4f}  |  {labels[idx]}')

## Observations

- Sentences within the **same topic** (e.g., Cricket with Cricket) show high cosine similarity (typically 0.55–0.85).
- Sentences from **different topics** show low similarity (typically 0.05–0.25), demonstrating that the model has captured semantic domain boundaries.
- The query *"The bowler took three wickets in one over"* correctly identifies Cricket sentences as the top-2 matches — specifically sentences about **wickets** — confirming that the model understands cricket-specific vocabulary and context, not just keyword overlap.